# Portfolio Lab — EUR investor study

Run top to bottom after package installation. All calculations and safeguards live in portfolio_lab. No prices or FX are forward-filled.

In [ ]:
from dataclasses import replace
from pathlib import Path
from IPython.display import display, Image
from portfolio_lab.config import load_config, configure_universe
from portfolio_lab.data import load_market_data, study_coverage
from portfolio_lab.leverage import add_leverage
from portfolio_lab.metrics import compute_asset_metrics
from portfolio_lab.optimization import optimize_frontier
from portfolio_lab.plotting import plot_frontier, plot_drawdown, allocation_summary
from portfolio_lab.reporting import write_review_outputs

project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


## User configuration
Edit the universe and leverage below. Metadata and calculations use this single effective configuration. Run All after every change.

In [ ]:
# Study universe: edit these metadata entries, then Run All.
assets = {'Apple': {'symbol': 'AAPL',
           'instrument': 'Primary US equity',
           'currency': 'USD',
           'return_type': 'Adjusted price',
           'distributions': 'Yahoo split/dividend adjustment; reinvestment proxy before investor tax',
           'limitation': 'Adjusted prices are a vendor proxy, not an independently audited TR index.',
           'reference': ''},
 'Nasdaq 100': {'symbol': 'QQQ',
                'instrument': 'Invesco QQQ ETF; Nasdaq-100 exposure proxy',
                'currency': 'USD',
                'return_type': 'Adjusted price',
                'distributions': 'Yahoo dividend/split adjustments; distributions assumed reinvested',
                'limitation': 'Exact Nasdaq-100 TR unavailable from tested Yahoo ^XNDX. QQQ is an ETF proxy '
                              'with fund fees/tracking differences, not the gross TR benchmark.',
                'reference': 'https://www.invesco.com/qqq-etf/en/home.html'},
 'S&P 500': {'symbol': '^SP500TR',
             'instrument': 'S&P 500 Total Return index',
             'currency': 'USD',
             'return_type': 'Gross Return',
             'distributions': 'Gross dividends reinvested by index methodology',
             'limitation': '',
             'reference': 'https://www.spglobal.com/spdji/en/indices/equity/sp-500/'},
 'Dow Jones': {'symbol': 'DIA',
               'instrument': 'State Street SPDR Dow Jones Industrial Average ETF proxy',
               'currency': 'USD',
               'return_type': 'Adjusted price',
               'distributions': 'Yahoo dividend/split adjustments; distributions assumed reinvested',
               'limitation': 'Exact Dow Jones TR unavailable from tested Yahoo ^DJITR. DIA is an ETF proxy '
                             'with fund fees/tracking differences, not the gross TR benchmark.',
               'reference': 'https://www.ssga.com/us/en/individual/etfs/state-street-spdr-dow-jones-industrial-average-etf-trust-dia'},
 'CAC 40': {'symbol': 'PX1GR.PA',
            'instrument': 'CAC 40 Gross Return Index',
            'currency': 'EUR',
            'return_type': 'Gross Return',
            'distributions': 'Gross dividends reinvested in index levels; no additional dividend adjustment',
            'limitation': 'Yahoo daily history has missing observations; inspect raw valid end and missing '
                          'counts before interpreting coverage.',
            'reference': 'https://live.euronext.com/en/product/indices/QS0011131834-XPAR'},
 'Gold': {'symbol': 'GC=F',
          'instrument': 'Yahoo continuous COMEX gold futures quotation',
          'currency': 'USD',
          'return_type': 'Price Return',
          'distributions': 'No dividends; quoted futures price changes only',
          'limitation': 'Not spot gold or an investable futures total-return index: roll, collateral yield '
                        'and contract stitching are unmodeled.',
          'reference': ''}}

# Leverage entries must reference names present in assets.
leveraged_assets = {"Apple": 1.5, "Nasdaq 100": 2.0}

# Examples (uncomment, adjust metadata, then Run All):
# del assets["Dow Jones"]
# assets["Apple"]["symbol"] = "AAPL"
# assets["MSCI World"] = dict(symbol="IWDA.AS", currency="EUR",
#     return_type="Adjusted price", instrument="iShares Core MSCI World UCITS ETF",
#     distributions="Accumulating ETF; income reinvested within the fund")
# If removing Apple: del assets["Apple"]; del leveraged_assets["Apple"]

config = configure_universe(load_config(project_dir), assets, leveraged_assets)
# Optional study overrides, applied to the same effective configuration:
# config = replace(config, risk_free_rate=0.02, frontier_points=75)


## Market data and provenance

Adjusted equity prices are a reinvestment proxy. Gross-return indices incorporate gross dividends. Gold futures quotations are not an investable total-return benchmark.

In [ ]:
data = load_market_data(config)
display(data.provenance)
display(data.fx_sample)
display(study_coverage(data, config))


## Daily-reset synthetic assets

Reset on every native underlying session, then translate NAV into EUR with unlevered FX. Compound before sampling common dates.

In [ ]:
prices, leverage_diagnostics = add_leverage(data.common, data.native, data.fx, config)
display(leverage_diagnostics)

## Asset performance in EUR

In [ ]:
stats = compute_asset_metrics(prices, config.trading_days, config.risk_free_rate, config.calendar_days_per_year)
display(stats)

## Growth frontier and volatility ceilings

Constant weights, long-only; rebalance at each common observation. Leveraged versions replace their underlying in the configured universe.

In [ ]:
optimization = optimize_frontier(prices, config)
display(optimization.summary)
display(optimization.weights)
display(allocation_summary(optimization.weights, config.allocation_display_threshold))
display(optimization.dominance)

## Charts

In [ ]:
frontier_figure = plot_frontier(stats, optimization)
drawdown_figure = plot_drawdown(stats)

## Export evidence and validate numerical results

In [ ]:
sanity = write_review_outputs(data, prices, stats, leverage_diagnostics, optimization, config)
frontier_figure.savefig(config.output_dir / "frontier.png", dpi=150)
drawdown_figure.savefig(config.output_dir / "drawdown.png", dpi=150)
display(sanity)
display(Image(filename=str(config.output_dir / "frontier.png")))
display(Image(filename=str(config.output_dir / "drawdown.png")))
